# 21 · ALS Matrix Factorization (Implicit Feedback)

**Purpose.** The first model that *learns representations* instead of counting.
Item-kNN failed the discovery test because it can only connect items that
literally co-occur in baskets. Matrix factorization can connect items that
have **never appeared together** but are bought by *similar households* —
similarity flows through the latent space, not the co-occurrence table. This
notebook tests whether that generalization beats popularity on the discovery
slice, and produces embeddings that later serve as ranker features and a
candidate source regardless.

## Theory

**The object.** Approximate the household×product interaction matrix
R (≈2.5K × ~90K, >99% empty) as a product of two thin matrices:

    R ≈ U · Vᵀ        U: households × f,   V: products × f,   f ≈ 64

Each household and each product becomes a learned f-dimensional vector
(**embedding**); the predicted affinity of household u for product i is the
dot product u·v. Recommendation = rank products by dot product.

**Why "implicit" is its own problem.** We observe purchases, not ratings.
A purchase signals preference; an *absence* is ambiguous (dislike? never seen?
out of stock?). The implicit-ALS formulation (Hu, Koren & Volinsky 2008)
treats every cell as a 0/1 preference with a **confidence weight**:
observed cells get confidence 1 + α·(interaction strength); unobserved cells
participate as weak zeros. The loss is weighted reconstruction over the
*entire* matrix — not just observed entries — which is what makes the
"weak zero" trick work.

**Why ALS.** The loss is biconvex: fix V and solving for U is ridge
regression per household; fix U and V solves per product. Alternate to
convergence — embarrassingly parallel, no learning rate to tune.

**Knobs.** `factors` (embedding dim), `regularization` (ridge λ),
`alpha` (confidence scale on interaction counts), `iterations`.
Defaults first; no tuning until the evaluation says it's worth it.

**Leakage control.** The matrix is built exclusively from transactions with
`day_no ≤ as_of`. Evaluation protocol identical to notebooks 10/20 —
total recall + the discovery slice, @10, vs oracle ceiling 0.402.

In [1]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"   # implicit's guidance: avoid thread thrashing

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import scipy.sparse as sp
from pathlib import Path

from implicit.als import AlternatingLeastSquares
from retail_ds.evaluate.metrics import recall_at_k, hit_rate_at_k, ndcg_at_k

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parents[1]

CFG = yaml.safe_load((ROOT / "configs" / "base.yaml").read_text())
con = duckdb.connect((ROOT / "db" / "retail.duckdb").as_posix(), read_only=True)

def q(sql: str) -> pd.DataFrame:
    return con.sql(sql).df()

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 4.5)

AS_OF = CFG["snapshots"]["valid"]   # 600
K = CFG["top_k"]                    # 10
labels = q(f"SELECT household_key, product_id FROM purchase_labels({AS_OF}, 30)")

# discovery labels, same definition as notebook 20
labels_new = q(f"""
    SELECT l.household_key, l.product_id
    FROM purchase_labels({AS_OF}, 30) l
    LEFT JOIN household_product_snapshot({AS_OF}) h
      USING (household_key, product_id)
    WHERE h.product_id IS NULL
""")
rows = []

print("as_of:", AS_OF, "| label households:", labels["household_key"].nunique(),
      "| discovery share:", f"{len(labels_new)/len(labels):.1%}")

C:\Users\siava\Recommendation_system\retail-recsys-platform\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


as_of: 600 | label households: 2054 | discovery share: 53.2%


In [2]:
interactions = q(f"""
    SELECT household_key, product_id, COUNT(DISTINCT basket_id) AS times_bought
    FROM staging.stg_transactions
    WHERE day_no <= {AS_OF}
    GROUP BY household_key, product_id
""")

households = np.sort(interactions["household_key"].unique())
products   = np.sort(interactions["product_id"].unique())
hh_index   = {h: i for i, h in enumerate(households)}
prod_index = {p: i for i, p in enumerate(products)}

matrix = sp.csr_matrix(
    (interactions["times_bought"].astype(np.float32),
     (interactions["household_key"].map(hh_index),
      interactions["product_id"].map(prod_index))),
    shape=(len(households), len(products)))

print(f"matrix: {matrix.shape[0]:,} x {matrix.shape[1]:,}, "
      f"density {matrix.nnz / (matrix.shape[0] * matrix.shape[1]):.4%}")

matrix: 2,498 x 84,180, density 0.5681%


In [3]:
als = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=20.0,
                              iterations=20, random_state=42)
als.fit(matrix)

100%|██████████| 20/20 [00:04<00:00,  4.39it/s]


In [4]:
def recs_frame(ids):
    return pd.DataFrame({
        "household_key": np.repeat(households, K),
        "product_id":    products[ids.ravel()],
        "rank":          np.tile(np.arange(1, K + 1), len(households)),
    })

all_users = np.arange(len(households))
ids_mixed, _ = als.recommend(all_users, matrix, N=K, filter_already_liked_items=False)
ids_new,   _ = als.recommend(all_users, matrix, N=K, filter_already_liked_items=True)

als_mixed = recs_frame(ids_mixed)
als_new   = recs_frame(ids_new)

print("TOTAL     — als_mixed :",
      f"recall@{K} {recall_at_k(als_mixed, labels, K):.3f}",
      f"| hit_rate {hit_rate_at_k(als_mixed, labels, K):.3f}",
      f"| ndcg {ndcg_at_k(als_mixed, labels, K):.3f}")
print("DISCOVERY — als_new   :",
      f"recall@{K} {recall_at_k(als_new, labels_new, K):.3f}",
      f"| hit_rate {hit_rate_at_k(als_new, labels_new, K):.3f}")

TOTAL     — als_mixed : recall@10 0.023 | hit_rate 0.263 | ndcg 0.043
DISCOVERY — als_new   : recall@10 0.008 | hit_rate 0.129


In [5]:
matrix_log = matrix.copy()
matrix_log.data = np.log1p(matrix_log.data).astype(np.float32)

als_log = AlternatingLeastSquares(factors=64, regularization=0.05, alpha=20.0,
                                  iterations=20, random_state=42)
als_log.fit(matrix_log)

ids_mixed2, _ = als_log.recommend(all_users, matrix_log, N=K, filter_already_liked_items=False)
ids_new2,   _ = als_log.recommend(all_users, matrix_log, N=K, filter_already_liked_items=True)

print("TOTAL     — als_log_mixed:", f"{recall_at_k(recs_frame(ids_mixed2), labels, K):.3f}",
      f"| hit {hit_rate_at_k(recs_frame(ids_mixed2), labels, K):.3f}")
print("DISCOVERY — als_log_new  :", f"{recall_at_k(recs_frame(ids_new2), labels_new, K):.3f}",
      f"| hit {hit_rate_at_k(recs_frame(ids_new2), labels_new, K):.3f}")

100%|██████████| 20/20 [00:04<00:00,  4.61it/s]


TOTAL     — als_log_mixed: 0.027 | hit 0.394
DISCOVERY — als_log_new  : 0.010 | hit 0.171


**Findings — ALS (as-of 600, K=10, factors=64, log1p confidence, untuned).**
Discovery slice: als_log_new 0.010 / 0.171 vs popularity 0.007 / 0.094 —
the first and only model to beat popularity on discovery (+43% recall, +82%
hit-rate relative); embeddings reach items that never co-occurred, which
kNN structurally cannot. Total slice: als_log_mixed 0.027 / 0.394 — still far
below buy-again (0.072), as expected; raw times-bought weighting was worse on
both slices (log1p scaling lifted total hit-rate 0.26 → 0.39), confirming
heavy staples were dominating the embeddings.

**Decisions:** ALS-log (new-only) takes the discovery candidate-generator
seat; ALS embeddings graduate to the ranker as features (household·product
affinity); repeat side stays with buy-again. Ranker architecture fixed:
buy-again ∪ popularity ∪ ALS-new candidates → XGBoost. No further ALS tuning
unless the candidate audit shows discovery recall is the binding constraint.

In [6]:
als_log_mixed = recs_frame(ids_mixed2)

board = pd.read_csv(ROOT / "reports" / "leaderboard.csv", index_col="model")
als_board = pd.DataFrame([
    {"model": "als_log_mixed",
     f"recall@{K}": recall_at_k(als_log_mixed, labels, K),
     f"hit_rate@{K}": hit_rate_at_k(als_log_mixed, labels, K),
     f"ndcg@{K}": ndcg_at_k(als_log_mixed, labels, K)},
]).set_index("model").round(3)
als_board["share_of_ceiling"] = (als_board[f"recall@{K}"] / 0.402).round(2)
pd.concat([board, als_board]).sort_values(f"recall@{K}", ascending=False)\
  .to_csv(ROOT / "reports" / "leaderboard.csv")